# Tp3 Pipeline Rag Simple - CORRECTION

## Partie 1 : Installation et Configuration - CORRECTION

In [ ]:
# Installation
# !pip install langchain langchain-community chromadb sentence-transformers pypdf openai -q


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader, DirectoryLoader
import chromadb
from sentence_transformers import SentenceTransformer
import os
from pprint import pprint
from collections import Counter
import numpy as np

c:\Users\Administrateur\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Partie 2 : Préparation des Documents - CORRECTION

In [ ]:
# Créer un répertoire pour les documents
import os

docs_dir = "./tp3_documents"
os.makedirs(docs_dir, exist_ok=True)

# Documents d'exemple
documents = {
    "python_basics.txt": """
# Python : Les Fondamentaux

Python est un langage de programmation interprété, orienté objet et de haut niveau créé par Guido van Rossum en 1991.
Il se distingue par sa syntaxe claire et lisible qui favorise la productivité des développeurs.

## Types de Données

Python propose plusieurs types de données natifs :
- **Numériques** : int (entiers), float (nombres à virgule), complex (nombres complexes)
- **Séquences** : str (chaînes), list (listes), tuple (tuples immuables)
- **Mappings** : dict (dictionnaires)
- **Ensembles** : set (ensembles), frozenset (ensembles immuables)
- **Booléens** : bool (True/False)

## Structures de Contrôle

Les structures de contrôle en Python incluent :
- **Conditionnelles** : if, elif, else
- **Boucles** : for (itération), while (condition)
- **Gestion d'erreurs** : try, except, finally

## Fonctions

Les fonctions en Python sont définies avec le mot-clé `def`. Elles peuvent accepter des arguments positionnels,
des arguments nommés, des arguments par défaut, et même un nombre variable d'arguments (*args, **kwargs).

Exemple :
```python
def calculer_moyenne(*nombres):
    return sum(nombres) / len(nombres) if nombres else 0
```

## Modules et Packages

Python dispose d'un vaste écosystème de bibliothèques. La bibliothèque standard inclut des modules pour :
- Manipulation de fichiers (os, pathlib)
- Calculs mathématiques (math, statistics)
- Dates et heures (datetime)
- Expressions régulières (re)
- Requêtes HTTP (urllib, http)
""",
    "machine_learning.txt": """
# Introduction au Machine Learning

Le Machine Learning (apprentissage automatique) est une branche de l'intelligence artificielle qui permet
aux ordinateurs d'apprendre à partir de données sans être explicitement programmés pour chaque tâche.

## Types d'Apprentissage

### Apprentissage Supervisé

Dans l'apprentissage supervisé, le modèle est entraîné sur des données étiquetées. Les algorithmes courants incluent :
- **Régression Linéaire** : prédiction de valeurs continues
- **Régression Logistique** : classification binaire
- **Arbres de Décision** : classification et régression
- **Random Forest** : ensemble d'arbres de décision
- **SVM (Support Vector Machines)** : classification avec marge maximale
- **Réseaux de Neurones** : modèles inspirés du cerveau humain

### Apprentissage Non Supervisé

L'apprentissage non supervisé travaille avec des données non étiquetées :
- **K-Means** : clustering par centres
- **DBSCAN** : clustering basé sur la densité
- **PCA (Analyse en Composantes Principales)** : réduction de dimensionnalité
- **Autoencoders** : apprentissage de représentations

### Apprentissage par Renforcement

Un agent apprend à prendre des décisions en interagissant avec un environnement et en recevant des récompenses.
Algorithmes populaires : Q-Learning, Deep Q-Networks (DQN), Policy Gradients.

## Bibliothèques Python

Les principales bibliothèques Python pour le ML sont :
- **scikit-learn** : algorithmes classiques de ML
- **TensorFlow** : deep learning développé par Google
- **PyTorch** : deep learning développé par Facebook
- **Keras** : API haut niveau pour les réseaux de neurones
- **XGBoost** : gradient boosting optimisé

## Workflow Typique

1. **Collecte de données** : rassembler les données pertinentes
2. **Exploration (EDA)** : analyser et visualiser les données
3. **Prétraitement** : nettoyage, normalisation, feature engineering
4. **Séparation** : train/validation/test sets
5. **Entraînement** : ajuster le modèle sur les données d'entraînement
6. **Évaluation** : mesurer les performances sur les données de test
7. **Optimisation** : hyperparamètres, architecture
8. **Déploiement** : mise en production du modèle
""",
    "rag_systems.txt": """
# RAG : Retrieval-Augmented Generation

RAG (Retrieval-Augmented Generation) est une architecture qui combine la recherche d'information (retrieval)
avec la génération de texte par des modèles de langage (generation).

## Principe de Fonctionnement

Un système RAG fonctionne en plusieurs étapes :

1. **Indexation** (offline) :
   - Découpage des documents en chunks
   - Génération d'embeddings vectoriels
   - Stockage dans une base de données vectorielle

2. **Recherche** (online) :
   - Conversion de la question en embedding
   - Recherche des documents les plus pertinents
   - Récupération du contexte

3. **Génération** (online) :
   - Construction d'un prompt avec le contexte
   - Génération de la réponse par le LLM
   - Post-traitement et validation

## Composants Clés

### Chunking (Découpage)

Le découpage des documents est crucial pour la performance :
- **Taille des chunks** : généralement 500-1000 tokens
- **Overlap** : chevauchement de 10-20% pour la continuité
- **Méthodes** : par caractères, par phrases, par paragraphes, sémantique

### Embeddings

Les modèles d'embedding transforment le texte en vecteurs :
- **OpenAI Embeddings** : text-embedding-3-small/large
- **Open Source** : all-MiniLM-L6-v2, BGE, E5
- **Multilingues** : multilingual-e5-large

### Bases Vectorielles

Stockage et recherche efficace des embeddings :
- **ChromaDB** : simple et embarquée
- **Pinecone** : cloud, scalable
- **Weaviate** : open source avec fonctionnalités avancées
- **FAISS** : bibliothèque Facebook, très rapide
- **Qdrant** : haute performance avec filtrage

### LLMs (Modèles de Langage)

Génération de réponses contextualisées :
- **Propriétaires** : GPT-4, Claude, Gemini
- **Open Source** : Llama 2/3, Mistral, Mixtral
- **Spécialisés** : modèles fine-tunés pour domaines spécifiques

## Techniques Avancées

### Hybrid Search

Combinaison de recherche vectorielle et keyword-based (BM25) pour améliorer la pertinence.

### Reranking

Utilisation d'un modèle de reranking (ex: cross-encoders) pour affiner les résultats de recherche.

### Query Transformation

Amélioration des requêtes :
- **HyDE** : génération de documents hypothétiques
- **Multi-Query** : génération de plusieurs variations de la question
- **Step-back** : questions plus générales pour contexte additionnel

### Metadata Filtering

Utilisation de métadonnées pour filtrer les résultats (date, catégorie, source, etc.).

## Métriques d'Évaluation

### Retrieval
- **Recall@K** : proportion de documents pertinents récupérés
- **Precision@K** : proportion de documents récupérés qui sont pertinents
- **MRR (Mean Reciprocal Rank)** : rang moyen du premier document pertinent

### Generation
- **Faithfulness** : fidélité au contexte récupéré
- **Answer Relevancy** : pertinence de la réponse à la question
- **Context Relevancy** : pertinence du contexte récupéré

## Cas d'Usage

- **Documentation technique** : réponses sur documentation produit
- **Support client** : chatbots avec base de connaissances
- **Recherche juridique** : recherche dans corpus de lois
- **Médical** : aide au diagnostic basée sur littérature médicale
- **Éducation** : assistants pédagogiques personnalisés
""",
    "docker_kubernetes.txt": """
# Docker et Kubernetes : Guide Pratique

## Docker

Docker est une plateforme de conteneurisation qui permet d'empaqueter des applications avec toutes leurs dépendances.

### Concepts Fondamentaux

**Image** : Template immuable contenant l'application et ses dépendances.
- Créée à partir d'un Dockerfile
- Stockée dans un registry (Docker Hub, etc.)
- Versionnable avec des tags

**Container** : Instance en cours d'exécution d'une image.
- Isolé du système hôte
- Léger (partage le kernel)
- Éphémère par défaut

**Volume** : Stockage persistant pour les données.
- Survit à la suppression du conteneur
- Peut être partagé entre conteneurs

**Network** : Réseau virtuel pour la communication entre conteneurs.

### Commandes Docker Essentielles

```bash
# Images
docker build -t mon-app:v1 .        # Construire une image
docker images                        # Lister les images
docker pull nginx:latest             # Télécharger une image
docker push mon-app:v1               # Pousser vers registry

# Conteneurs
docker run -d -p 8080:80 nginx       # Lancer un conteneur
docker ps                            # Lister conteneurs actifs
docker ps -a                         # Tous les conteneurs
docker stop <container-id>           # Arrêter
docker rm <container-id>             # Supprimer
docker logs <container-id>           # Voir les logs
docker exec -it <container-id> bash  # Shell interactif

# Volumes
docker volume create mon-volume
docker run -v mon-volume:/app/data nginx
```

### Dockerfile

Structure type d'un Dockerfile :
```dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY . .
EXPOSE 8000
CMD ["python", "app.py"]
```

### Docker Compose

Orchestration multi-conteneurs avec un fichier YAML :
```yaml
version: '3.8'
services:
  web:
    build: .
    ports:
      - "8000:8000"
    volumes:
      - .:/app
    environment:
      - DEBUG=1
  db:
    image: postgres:15
    volumes:
      - postgres_data:/var/lib/postgresql/data
```

## Kubernetes

Kubernetes (K8s) est un orchestrateur de conteneurs open-source pour automatiser le déploiement,
la mise à l'échelle et la gestion d'applications conteneurisées.

### Architecture

**Control Plane** (Plan de contrôle) :
- **API Server** : point d'entrée pour toutes les opérations
- **etcd** : base de données clé-valeur pour l'état du cluster
- **Scheduler** : assigne les pods aux nodes
- **Controller Manager** : gère les contrôleurs (réplication, etc.)

**Nodes** (Nœuds de travail) :
- **kubelet** : agent sur chaque node
- **kube-proxy** : gestion du réseau
- **Container Runtime** : Docker, containerd, CRI-O

### Objets Kubernetes

**Pod** : Plus petite unité déployable, contient un ou plusieurs conteneurs.

**Deployment** : Gère le déploiement et la mise à jour des pods.
```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: mon-app
spec:
  replicas: 3
  selector:
    matchLabels:
      app: mon-app
  template:
    metadata:
      labels:
        app: mon-app
    spec:
      containers:
      - name: app
        image: mon-app:v1
        ports:
        - containerPort: 8000
```

**Service** : Expose les pods sur le réseau.
- ClusterIP : interne au cluster
- NodePort : exposé sur un port de chaque node
- LoadBalancer : avec load balancer externe

**ConfigMap** : Configuration non sensible.

**Secret** : Données sensibles (mots de passe, tokens).

**Ingress** : Gestion du trafic HTTP/HTTPS entrant.

### Commandes kubectl

```bash
# Déploiement
kubectl apply -f deployment.yaml
kubectl get deployments
kubectl get pods
kubectl describe pod <pod-name>

# Scaling
kubectl scale deployment mon-app --replicas=5

# Mise à jour
kubectl set image deployment/mon-app app=mon-app:v2
kubectl rollout status deployment/mon-app
kubectl rollout undo deployment/mon-app

# Logs et debug
kubectl logs <pod-name>
kubectl exec -it <pod-name> -- /bin/bash

# Services
kubectl get services
kubectl port-forward service/mon-app 8080:80
```

### Bonnes Pratiques

1. **Resource Limits** : définir requests et limits pour CPU/mémoire
2. **Health Checks** : liveness et readiness probes
3. **Rolling Updates** : mises à jour progressives sans downtime
4. **Namespaces** : isolation logique des ressources
5. **RBAC** : contrôle d'accès basé sur les rôles
6. **Monitoring** : Prometheus, Grafana pour la supervision
""",
    "data_science.txt": """
# Data Science avec Python

La Data Science combine statistiques, programmation et expertise métier pour extraire des insights des données.

## Bibliothèques Essentielles

### NumPy

Calcul numérique avec des arrays multidimensionnels :
```python
import numpy as np
arr = np.array([1, 2, 3, 4, 5])
matrix = np.array([[1, 2], [3, 4]])
```

Opérations : broadcasting, slicing, aggregations, algèbre linéaire.

### Pandas

Manipulation et analyse de données tabulaires :
```python
import pandas as pd
df = pd.read_csv('data.csv')
df.head()
df.describe()
df.groupby('category')['value'].mean()
```

Structures : Series (1D), DataFrame (2D).
Opérations : filtrage, groupby, merge, pivot, time series.

### Matplotlib et Seaborn

Visualisation de données :
```python
import matplotlib.pyplot as plt
import seaborn as sns

# Matplotlib
plt.plot(x, y)
plt.scatter(x, y)
plt.hist(data)

# Seaborn
sns.boxplot(x='category', y='value', data=df)
sns.heatmap(correlation_matrix)
```

### Scikit-learn

Machine Learning :
```python
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)
```

## Workflow Data Science

### 1. Acquisition des Données

Sources : CSV, bases de données, APIs, web scraping.
```python
df = pd.read_csv('data.csv')
# ou
import requests
response = requests.get('https://api.example.com/data')
```

### 2. Exploration (EDA)

Comprendre les données :
- Dimensions : shape, dtypes
- Statistiques : describe(), info()
- Valeurs manquantes : isnull().sum()
- Distributions : histogrammes, boxplots
- Corrélations : corr(), heatmap

### 3. Nettoyage

Préparer les données :
```python
# Valeurs manquantes
df.dropna()  # supprimer
df.fillna(df.mean())  # remplir

# Doublons
df.drop_duplicates()

# Types
df['date'] = pd.to_datetime(df['date'])

# Outliers
z_scores = (df['value'] - df['value'].mean()) / df['value'].std()
df = df[abs(z_scores) < 3]
```

### 4. Feature Engineering

Créer de nouvelles features :
```python
# Encodage catégoriel
df = pd.get_dummies(df, columns=['category'])

# Normalisation
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df)

# Binning
df['age_group'] = pd.cut(df['age'], bins=[0, 18, 35, 60, 100])
```

### 5. Modélisation

Entraîner et évaluer des modèles :
```python
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
```

### 6. Visualisation et Communication

Présenter les résultats :
- Graphiques interactifs : Plotly, Bokeh
- Dashboards : Streamlit, Dash
- Rapports : Jupyter Notebooks

## Outils Avancés

**Dask** : Pandas à grande échelle (parallélisation)
**PySpark** : Big Data avec Apache Spark
**MLflow** : Suivi d'expériences ML
**Great Expectations** : Validation de qualité des données
""",
}

# Écrire les fichiers
for filename, content in documents.items():
    filepath = os.path.join(docs_dir, filename)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content.strip())

print(f"✓ {len(documents)} documents créés dans {docs_dir}")
for filename in documents.keys():
    filepath = os.path.join(docs_dir, filename)
    size = os.path.getsize(filepath)
    print(f"  - {filename}: {size} bytes")

✓ 5 documents créés dans ./tp3_documents
  - python_basics.txt: 1551 bytes
  - machine_learning.txt: 2279 bytes
  - rag_systems.txt: 3361 bytes
  - docker_kubernetes.txt: 4576 bytes
  - data_science.txt: 3398 bytes


In [ ]:
# Charger les documents
loader = DirectoryLoader(
    docs_dir,
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)

documents = loader.load()

print(f"{len(documents)} documents chargés")
print(f"\nAperçu du premier document :")
print(f"Source : {documents[0].metadata['source']}")
print(f"Longueur : {len(documents[0].page_content)} caractères")
print(f"Début : {documents[0].page_content[:200]}...")

5 documents chargés

Aperçu du premier document :
Source : tp3_documents\data_science.txt
Longueur : 3222 caractères
Début : # Data Science avec Python

La Data Science combine statistiques, programmation et expertise métier pour extraire des insights des données.

## Bibliothèques Essentielles

### NumPy

Calcul numérique ...


## Partie 3 : Découpage en Chunks - CORRECTION

In [ ]:
# Découpage avec chunk_size = 500
text_splitter_500 = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=50, length_function=len, separators=["\n\n", "\n"]
)

chunks_500 = text_splitter_500.split_documents(documents)

print(f"Nombre de chunks : {len(chunks_500)}")

NameError: name 'RecursiveCharacterTextSplitter' is not defined

In [ ]:
# Comparer différentes tailles
chunk_sizes = [300, 500, 1000]
results = {}

for size in chunk_sizes:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=int(size * 0.1),  # 10% overlap
        length_function=len,
    )
    chunks = splitter.split_documents(documents)
    results[size] = chunks

    print(f"\nChunk size = {size}")
    print(f"  Nombre de chunks : {len(chunks)}")
    print(f"  Longueur moyenne : {np.mean([len(c.page_content) for c in chunks]):.1f}")
    print(f"  Longueur min : {min([len(c.page_content) for c in chunks])}")
    print(f"  Longueur max : {max([len(c.page_content) for c in chunks])}")

In [ ]:
# Analyse détaillée pour chunk_size = 500 (choisi comme optimal)
chunks = results[500]

# Statistiques
lengths = [len(c.page_content) for c in chunks]
print(f"\nNombre total de chunks : {len(chunks)}")
print(f"Longueur moyenne : {np.mean(lengths):.1f} caractères")
print(f"Écart-type : {np.std(lengths):.1f}")
print(f"Médiane : {np.median(lengths):.1f}")

# Distribution par document source
sources = [c.metadata["source"] for c in chunks]
source_counts = Counter(sources)
print("\nDistribution par document :")
for source, count in source_counts.most_common():
    filename = os.path.basename(source)
    print(f"  {filename}: {count} chunks")

# Exemples de chunks
for i in [0, len(chunks) // 2, len(chunks) - 1]:
    print(f"\nChunk #{i}")
    print(f"Source : {os.path.basename(chunks[i].metadata['source'])}")
    print(f"Longueur : {len(chunks[i].page_content)} caractères")
    print(f"Contenu : {chunks[i].page_content[:200]}...")
    print("-" * 80)

## Partie 4 : Indexation dans ChromaDB - CORRECTION

In [ ]:
# Créer le client et la collection
client = chromadb.Client()

# Supprimer si existe
try:
    client.delete_collection(name="docs_tech")
except:
    pass

# Créer la collection
collection = client.create_collection(
    name="docs_tech", metadata={"description": "Documentation technique pour RAG"}
)

print(f"✓ Collection '{collection.name}' créée")

In [ ]:
# Indexer les chunks
# Préparer les données
texts = [chunk.page_content for chunk in chunks]
ids = [f"chunk_{i}" for i in range(len(chunks))]
metadatas = [
    {
        "source": os.path.basename(chunk.metadata["source"]),
        "chunk_index": i,
        "length": len(chunk.page_content),
    }
    for i, chunk in enumerate(chunks)
]

# Ajouter par batches (ChromaDB recommande des batches de ~41666)
batch_size = 100
for i in range(0, len(texts), batch_size):
    batch_texts = texts[i : i + batch_size]
    batch_ids = ids[i : i + batch_size]
    batch_metadatas = metadatas[i : i + batch_size]

    collection.add(documents=batch_texts, ids=batch_ids, metadatas=batch_metadatas)
    print(f"  Batch {i // batch_size + 1}: {len(batch_texts)} chunks ajoutés")

print(f"\n{collection.count()} chunks indexés dans ChromaDB")

## Partie 5 : Création du Retriever - CORRECTION

In [ ]:
# Fonction de recherche
def retrieve(question, top_k=3, show_details=True):
    # Recherche
    results = collection.query(query_texts=[question], n_results=top_k)

    # Formater les résultats
    formatted_results = []
    for i in range(len(results["documents"][0])):
        result = {
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i],
            "similarity": 1 - results["distances"][0][i],
            "id": results["ids"][0][i],
        }
        formatted_results.append(result)

    # Afficher si demandé
    if show_details:
        print(f"\nQuestion : {question}")
        print("=" * 80)
        for i, result in enumerate(formatted_results, 1):
            print(
                f"\n[{i}] Similarité: {result['similarity']:.4f} | Source: {result['metadata']['source']}"
            )
            print(f"Texte (300 premiers caractères): {result['text'][:300]}...")

    return formatted_results


In [ ]:
# Tests du retriever
test_questions = [
    "Comment fonctionne Docker ?",
    "Qu'est-ce que le RAG ?",
    "Quelles sont les bibliothèques Python pour le machine learning ?",
]

for question in test_questions:
    results = retrieve(question, top_k=3)
    print("\n" + "#" * 80 + "\n")

## Partie 6 : Génération de Réponses - CORRECTION

In [ ]:
# Template de prompt
def create_prompt(question, context_chunks):
    # Construire le contexte
    context = "\n\n".join(
        [
            f"[Document {i + 1} - {chunk['metadata']['source']}]\n{chunk['text']}"
            for i, chunk in enumerate(context_chunks)
        ]
    )

    # Template de prompt
    prompt = f"""Tu es un assistant technique expert qui répond aux questions basées sur la documentation fournie.

INSTRUCTIONS :
1. Réponds UNIQUEMENT en te basant sur le contexte fourni ci-dessous
2. Si l'information n'est pas dans le contexte, dis "Je ne trouve pas cette information dans la documentation fournie"
3. Sois précis et concis
4. Cite les documents sources quand c'est pertinent
5. Structure ta réponse avec des paragraphes et des listes si nécessaire

CONTEXTE :
{context}

QUESTION : {question}

RÉPONSE :"""

    return prompt

In [ ]:
# Fonction answer() complète
def answer(question, top_k=3, use_llm=False):
    # Étape 1 : Récupérer le contexte
    print(f"QUESTION : {question}")

    context_chunks = retrieve(question, top_k=top_k, show_details=False)

    print(f"{len(context_chunks)} chunks récupérés")
    for i, chunk in enumerate(context_chunks, 1):
        print(
            f"  {i}. {chunk['metadata']['source']} (similarité: {chunk['similarity']:.3f})"
        )

    # Étape 2 : Créer le prompt
    prompt = create_prompt(question, context_chunks)

    # Étape 3 : Générer la réponse
    if use_llm:
        try:
            import openai

            # Nécessite OPENAI_API_KEY dans l'environnement
            response = openai.ChatCompletion.create(
                model="gpt-3.5-turbo",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.3,
            )
            generated_answer = response.choices[0].message.content
        except Exception as e:
            generated_answer = f"Erreur lors de la génération avec LLM : {e}\n\nVeuillez configurer votre clé API OpenAI."
    else:
        generated_answer = "[Mode sans LLM] Veuillez utiliser le prompt ci-dessous avec votre LLM préféré."

    # Résultat
    result = {
        "question": question,
        "context": context_chunks,
        "prompt": prompt,
        "answer": generated_answer,
    }

    # Affichage

    for i, chunk in enumerate(context_chunks, 1):
        print(f"\n[{i}] {chunk['metadata']['source']}")
        print(chunk["text"][:300] + "...")

    print(prompt[:500] + "..." if len(prompt) > 500 else prompt)

    print("RÉPONSE :")

    print(generated_answer)

    return result


## Partie 7 : Tests et Évaluation - CORRECTION

In [ ]:
# 10 questions de test
evaluation_questions = [
    # Python
    "Quels sont les types de données disponibles en Python ?",
    "Comment définir une fonction en Python ?",
    # Machine Learning
    "Quelle est la différence entre apprentissage supervisé et non supervisé ?",
    "Quelles bibliothèques Python utiliser pour le machine learning ?",
    # RAG
    "Qu'est-ce que RAG et comment ça fonctionne ?",
    "Quelles sont les techniques avancées pour améliorer un système RAG ?",
    # Docker/Kubernetes
    "Comment créer une image Docker ?",
    "Quelle est l'architecture de Kubernetes ?",
    # Data Science
    "Quelles sont les étapes du workflow data science ?",
    "Comment gérer les valeurs manquantes en Pandas ?",
]

print(f"{len(evaluation_questions)} questions de test préparées")

In [ ]:
# Évaluation complète


evaluation_results = []

for i, question in enumerate(evaluation_questions, 1):
    print(f"TEST {i}/{len(evaluation_questions)}")

    result = answer(question, top_k=3, use_llm=False)
    evaluation_results.append(result)

    # Pause entre les questions pour la lisibilité
    import time

    time.sleep(1)


In [ ]:
avg_similarities = []
for result in evaluation_results:
    similarities = [chunk["similarity"] for chunk in result["context"]]
    avg_sim = np.mean(similarities)
    avg_similarities.append(avg_sim)

print(
    f"Similarité moyenne (top-1) : {np.mean([r['context'][0]['similarity'] for r in evaluation_results]):.3f}"
)
print(f"Similarité moyenne (top-3) : {np.mean(avg_similarities):.3f}")
print(f"Similarité min : {min(avg_similarities):.3f}")
print(f"Similarité max : {max(avg_similarities):.3f}")

# Distribution des sources

all_sources = []
for result in evaluation_results:
    for chunk in result["context"]:
        all_sources.append(chunk["metadata"]["source"])

source_counts = Counter(all_sources)
for source, count in source_counts.most_common():
    print(f"  {source}: {count} fois récupéré")
